In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch11-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "11-05-gnn-explanations.py"

# 遮去原子的資料，或擋住一條鍵上的訊息

以實測的 ESOL 水溶解度訓練模型。Colab 首次執行會下載資料。以下分析在獨立的 Python 程序中執行；試做練習時，可把程式接在分析格末端，再重跑分析格與顯示格。

In [ ]:
%%writefile "{script}"
from collections import defaultdict
from pathlib import Path
from io import BytesIO
import os

import deepchem as dc
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

# Delaney ESOL，實測 log10(S / (mol/L))。DOI 10.1021/ci034243x
_, (data,), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
observed = defaultdict(list)
for smi, y, w in zip(data.ids, data.y[:, 0], data.w[:, 0]):
    if w > 0 and np.isfinite(y):
        observed[Chem.MolToSmiles(Chem.MolFromSmiles(smi))].append(float(y))
train, test = train_test_split(sorted(observed), test_size=0.2, random_state=42)
def labels(smiles):
    return np.array([np.mean(observed[s]) for s in smiles])

target = "COc1ccc(Cl)cc1"  # 對氯苯甲醚
assert target in test
mol = Chem.MolFromSmiles(target)
print("train / test:", len(train), len(test), "| observed log S:", labels([target])[0])

def structure(atom_colors, bond_colors):
    drawing = rdMolDraw2D.MolDraw2DCairo(600, 380)
    drawing.drawOptions().addAtomIndices = True
    drawing.drawOptions().useBWAtomPalette()
    drawing.DrawMolecule(mol, highlightAtoms=list(atom_colors), highlightBonds=list(bond_colors),
                         highlightAtomColors=atom_colors, highlightBondColors=bond_colors)
    drawing.FinishDrawing()
    return Image.open(BytesIO(drawing.GetDrawingText())).convert("RGB")

out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
from scipy.optimize import minimize

# 一層圖網路。每個原子都用同一組權重讀取自己的特徵和鄰居的特徵總和。
elements = [6, 7, 8, 9, 15, 16, 17, 35, 53]
def graph_inputs(molecule, hide_atom=None, hide_bond=None):
    x = []
    for atom in molecule.GetAtoms():
        z = atom.GetAtomicNum()
        x.append([float(z == e) for e in elements] + [float(z not in elements),
                 float(atom.GetIsAromatic()), float(atom.GetFormalCharge())])
    x = np.asarray(x)
    if hide_atom is not None:
        x[hide_atom] = 0
    neighbor_sum = np.zeros_like(x)
    for bond in molecule.GetBonds():
        if bond.GetIdx() == hide_bond:
            continue
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        neighbor_sum[i] += x[j]
        neighbor_sum[j] += x[i]
    return np.column_stack([x, neighbor_sum])

# 一次串接所有訓練原子，owner 記錄各列屬於哪個分子。
rows = [graph_inputs(Chem.MolFromSmiles(s)) for s in train]
features = np.vstack(rows)
owner = np.repeat(np.arange(len(rows)), [len(m) for m in rows])
y = labels(train)
width = 8
cut = features.shape[1] * width

def unpack(parameters):
    return parameters[:cut].reshape(-1, width), parameters[cut:-1], parameters[-1]

def objective(parameters):
    W, v, bias = unpack(parameters)
    h = np.tanh(features @ W)
    prediction = bias + np.bincount(owner, weights=h @ v)
    residual = prediction - y
    d = 2 * residual / len(y)
    atom_d = d[owner]
    penalty = 0.01
    loss = np.mean(residual**2) + penalty * np.sum(parameters[:-1]**2)
    dW = features.T @ ((atom_d[:, None] * v) * (1 - h**2))
    gradient = np.r_[dW.ravel(), h.T @ atom_d, d.sum()]
    gradient[:-1] += 2 * penalty * parameters[:-1]
    return loss, gradient

rng = np.random.default_rng(42)
start = rng.normal(0, 0.12, cut + width + 1)
fit = minimize(objective, start, jac=True, method="L-BFGS-B",
               options={"maxiter": 300, "ftol": 1e-8})
W, v, bias = unpack(fit.x)
def predict(molecule, hide_atom=None, hide_bond=None):
    inputs = graph_inputs(molecule, hide_atom, hide_bond)
    return bias + (np.tanh(inputs @ W) @ v).sum()

full = predict(mol)
reverse = Chem.RenumberAtoms(mol, list(reversed(range(mol.GetNumAtoms()))))
assert np.isclose(predict(reverse), full)
print("iterations / training objective:", fit.nit, fit.fun)
print("test RMSE:", root_mean_squared_error(labels(test), [predict(Chem.MolFromSmiles(s)) for s in test]))
atom_delta = np.array([full - predict(mol, hide_atom=i) for i in range(mol.GetNumAtoms())])
bond_delta = np.array([full - predict(mol, hide_bond=j) for j in range(mol.GetNumBonds())])
print("full:", full, "atom differences:", atom_delta.round(3))
for bond, delta in zip(mol.GetBonds(), bond_delta):
    print("bond / endpoints / difference:", bond.GetIdx(), bond.GetBeginAtomIdx(), bond.GetEndAtomIdx(), round(delta, 3))

scale = max(abs(np.r_[atom_delta, bond_delta]))
def colors(values):
    return {i: tuple(plt.get_cmap("RdBu_r")((1 + v / scale) / 2)[:3]) for i, v in enumerate(values)}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.3), layout="constrained")
axes[0].imshow(structure(colors(atom_delta), {}))
axes[0].set_title("Mask one atom's input features")
axes[1].imshow(structure({}, colors(bond_delta)))
axes[1].set_title("Block messages across one bond")
for ax in axes:
    ax.axis("off")
mappable = plt.cm.ScalarMappable(norm=plt.Normalize(-scale, scale), cmap="RdBu_r")
fig.colorbar(mappable, ax=axes, shrink=.74, label="Full minus masked (log S)")
fig.savefig(out / "11-05-gnn-explanations.png", dpi=170)


In [ ]:
from IPython.display import Image, display

import os
environment = dict(os.environ, VECLIB_MAXIMUM_THREADS="1", OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MPLBACKEND="Agg")
subprocess.run([python, str(script)], check=True, env=environment)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "11-05-gnn-explanations.png")))